# Active Learning Lab

Базовый экспериментальный ноутбук для:
- синтетических smoke-тестов;
- загрузки реальных датасетов из Hugging Face;
- локального цикла обучения без SDK;
- запуска через Active Learning SDK;
- сохранения короткой таблицы результатов рядом с ноутбуком.

Идея простая: копируешь этот ноутбук под новую гипотезу, меняешь только нужные места и сравниваешь результаты по общей таблице.

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import os
import random
import shutil
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, log_loss
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# cuBLAS workspace config helps PyTorch keep CUDA execution deterministic.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

def resolve_repo_root(start_path: Path) -> Path:
    """Find the repository root by walking upward until `src/active_learning_sdk` exists."""
    start_path = start_path.resolve()
    for candidate in (start_path, *start_path.parents):
        if (candidate / "src" / "active_learning_sdk").exists():
            return candidate
    return start_path


# Keep notebook imports stable even when the notebook is copied into a subfolder.
REPO_ROOT = resolve_repo_root(Path(os.environ.get("AL_LAB_REPO_ROOT", Path.cwd())))
OUTPUT_DIR = Path(os.environ.get("AL_LAB_OUTPUT_DIR", Path.cwd())).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    # Notebook lives outside `src`, so add the package root explicitly.
    sys.path.insert(0, str(SRC_DIR))

from active_learning_sdk import (
    ActiveLearningProject,
    AnnotationPolicy,
    CacheConfig,
    FingerprintConfig,
    LabelBackendConfig,
    LabelSchema,
    SchedulerConfig,
    SplitConfig,
    StopCriteria,
)
from active_learning_sdk.backends.base import RoundProgress, RoundPullResult, RoundPushResult
from active_learning_sdk.types import AnnotationRecord, DataSample

# Print the runtime context so runs are easier to compare.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={DEVICE}")
print(f"repo_root={REPO_ROOT}")
print(f"output_dir={OUTPUT_DIR}")


c:\Dev\active-learning-orchestrator\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device=cuda
repo_root=C:\Dev\active-learning-orchestrator
output_dir=C:\Dev\active-learning-orchestrator\lab


In [ ]:
# --- Run identity -------------------------------------------------------
RUN_NOTE = "TS vector local run"
PIPELINE_MODE = "local"  # local | sdk
STRATEGY_NAME = "random"  # random | entropy | margin | least_confidence
CALIBRATION_MODE = "vector"  # scalar | vector

# --- What to run --------------------------------------------------------
SELECTED_DATASETS = ["synthetic"]
SELECTED_MODELS = ["tiny_transformer"]

# --- Reproducibility and training budget --------------------------------
GLOBAL_SEED = 13
MAX_ROUNDS = 8
BATCH_SIZE = 16
TRAIN_EPOCHS = 3
TRAIN_BATCH_SIZE = 16
LEARNING_RATE = 3e-5
MAX_LENGTH = 128

# --- Result persistence --------------------------------------------------
SAVE_RESULTS = True

# Optional env overrides keep the same notebook scriptable from automation.
RUN_NOTE = os.environ.get("AL_LAB_RUN_NOTE", RUN_NOTE)
PIPELINE_MODE = os.environ.get("AL_LAB_PIPELINE_MODE", PIPELINE_MODE)
STRATEGY_NAME = os.environ.get("AL_LAB_STRATEGY", STRATEGY_NAME)
CALIBRATION_MODE = os.environ.get("AL_LAB_CALIBRATION_MODE", CALIBRATION_MODE)
if "AL_LAB_DATASETS" in os.environ:
    SELECTED_DATASETS = json.loads(os.environ["AL_LAB_DATASETS"])
if "AL_LAB_MODELS" in os.environ:
    SELECTED_MODELS = json.loads(os.environ["AL_LAB_MODELS"])
GLOBAL_SEED = int(os.environ.get("AL_LAB_SEED", str(GLOBAL_SEED)))
MAX_ROUNDS = int(os.environ.get("AL_LAB_MAX_ROUNDS", str(MAX_ROUNDS)))
BATCH_SIZE = int(os.environ.get("AL_LAB_BATCH_SIZE", str(BATCH_SIZE)))
TRAIN_EPOCHS = int(os.environ.get("AL_LAB_TRAIN_EPOCHS", str(TRAIN_EPOCHS)))
TRAIN_BATCH_SIZE = int(os.environ.get("AL_LAB_TRAIN_BATCH_SIZE", str(TRAIN_BATCH_SIZE)))
LEARNING_RATE = float(os.environ.get("AL_LAB_LR", str(LEARNING_RATE)))
MAX_LENGTH = int(os.environ.get("AL_LAB_MAX_LENGTH", str(MAX_LENGTH)))
SAVE_RESULTS = os.environ.get("AL_LAB_SAVE_RESULTS", "1" if SAVE_RESULTS else "0") == "1"

# These paths stay next to the notebook output directory by default.
RESULTS_TABLE_PATH = OUTPUT_DIR / "experiment_runs.csv"
HF_CACHE_DIR = OUTPUT_DIR / "_hf_cache"
SDK_RUNS_DIR = OUTPUT_DIR / "_sdk_runs"

# Synthetic config is the fastest place to shape toy-task difficulty.
SYNTHETIC_CONFIG = {
    "labels": ["sports", "business", "science", "culture"],
    "train_size": 240,
    "test_size": 120,
    "calibration_size": 60,
    "difficulty_weights": {"easy": 0.55, "medium": 0.3, "hard": 0.15},
    "label_noise": 0.02,
}

# Add new real datasets here and normalize them through one shared loader.
DATASET_REGISTRY = {
    "synthetic": {
        "kind": "synthetic",
    },
    "fancyzhx/ag_news": {
        "kind": "huggingface",
        "train_split": "train[:1600]",
        "calibration_split": "train[1600:2000]",
        "test_split": "test[:1000]",
        "expected_fingerprint": None,
    },
    "fancyzhx/dbpedia_14": {
        "kind": "huggingface",
        "train_split": "train[:1600]",
        "calibration_split": "train[1600:2000]",
        "test_split": "test[:1000]",
        "expected_fingerprint": None,
    },
    "community-datasets/yahoo_answers_topics": {
        "kind": "huggingface",
        "train_split": "train[:1600]",
        "calibration_split": "train[1600:2000]",
        "test_split": "test[:1000]",
        "expected_fingerprint": None,
    },
    "cornell-movie-review-data/rotten_tomatoes": {
        "kind": "huggingface",
        "train_split": "train[:512]",
        "calibration_split": "validation[:256]",
        "test_split": "test[:500]",
        "expected_fingerprint": None,
    },
    "SetFit/sst2": {
        "kind": "huggingface",
        "train_split": "train[:512]",
        "calibration_split": "validation[:256]",
        "test_split": "test[:500]",
        "expected_fingerprint": None,
    },
    "SetFit/TREC-QC": {
        "kind": "huggingface",
        "train_split": "train[:1024]",
        "calibration_split": "train[1024:1280]",
        "test_split": "test[:500]",
        "label_column": "label_coarse",
        "label_text_column": "label_coarse_text",
        "expected_fingerprint": None,
    },
}

# Keep one lightweight model and one realistic baseline available.
MODEL_REGISTRY = {
    "tiny_transformer": {
        "model_name": "distilbert-base-uncased",
        "freeze_encoder": False,
    },
    "roberta_base": {
        "model_name": "FacebookAI/roberta-base",
        "freeze_encoder": False,
    },
}

assert PIPELINE_MODE in {"local", "sdk"}
assert STRATEGY_NAME in {"random", "entropy", "margin", "least_confidence"}
assert CALIBRATION_MODE in {"scalar", "vector"}
print({
    "note": RUN_NOTE,
    "pipeline_mode": PIPELINE_MODE,
    "calibration_mode": CALIBRATION_MODE,
    "datasets": SELECTED_DATASETS,
    "models": SELECTED_MODELS,
    "strategy": STRATEGY_NAME,
    "save_results": SAVE_RESULTS,
})


{'note': 'baseline local run', 'pipeline_mode': 'local', 'datasets': ['synthetic'], 'models': ['tiny_transformer'], 'strategy': 'random', 'save_results': True}


In [3]:
def seed_everything(seed: int) -> None:
    """Seed Python, NumPy and Torch RNGs for reproducible runs."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def make_experiment_seed(*parts: Any) -> int:
    """Build a stable per-experiment seed from the global seed and run parts."""
    payload = "::".join(str(part) for part in parts)
    digest = hashlib.blake2b(payload.encode("utf-8"), digest_size=8).hexdigest()
    return (GLOBAL_SEED + int(digest, 16)) % (2**31)


# A small stable fingerprint helps compare runs across notebook copies.
def compute_dataset_fingerprint(frame: pd.DataFrame) -> str:
    """Hash a normalized dataset dataframe.

    Args:
        frame: DataFrame with `sample_id`, `text`, `label`, `split` columns.

    Returns:
        Stable hex digest used to compare dataset versions between runs.
    """
    columns = ["sample_id", "text", "label", "split"]
    payload = frame[columns].sort_values("sample_id").to_dict(orient="records")
    digest = hashlib.blake2b(digest_size=20)
    digest.update(json.dumps(payload, ensure_ascii=False, sort_keys=True).encode("utf-8"))
    return digest.hexdigest()


def _join_non_empty(parts: Sequence[Any]) -> str:
    """Join non-empty fragments into one clean text string."""
    return " ".join(str(part).strip() for part in parts if str(part).strip())


# Synthetic text is intentionally simple but still exposes easy/medium/hard cases.
def build_synthetic_dataset(config: Dict[str, Any], seed: int) -> tuple[pd.DataFrame, List[str], str]:
    """Create a small synthetic classification dataset.

    Args:
        config: Dict with `labels`, `train_size`, `test_size`,
            `difficulty_weights`, `label_noise`.
        seed: Integer seed for the local random generator.

    Returns:
        `(frame, label_names, fingerprint)` where `frame` contains
        `sample_id`, `text`, `label`, `label_text`, `split`,
        `source_dataset`, `difficulty`.

    Notes:
        Difficulty changes how strongly class-specific tokens dominate the text.
    """
    rng = random.Random(seed)
    label_names = list(config["labels"])
    # Keep a stable class-name -> integer-id mapping for labels stored in the frame.
    label_to_id = {name: index for index, name in enumerate(label_names)}

    class_words = {
        "sports": ["stadium", "coach", "league", "season", "playoff", "goal", "team", "match"],
        "business": ["market", "stocks", "merger", "finance", "profit", "trade", "bank", "revenue"],
        "science": ["research", "lab", "experiment", "discovery", "physics", "biology", "data", "study"],
        "culture": ["artist", "festival", "cinema", "music", "museum", "novel", "design", "theatre"],
    }
    common_words = ["today", "report", "global", "analysis", "update", "public", "official", "feature"]
    difficulty_weights = config["difficulty_weights"]
    difficulty_names = list(difficulty_weights.keys())
    difficulty_probs = list(difficulty_weights.values())

    def make_text(label_name: str, difficulty: str) -> str:
        """Generate one synthetic text from class-specific and noisy tokens."""
        own_words = class_words[label_name]
        # Harder samples include more tokens borrowed from the other classes.
        other_words = [word for key, values in class_words.items() if key != label_name for word in values]
        if difficulty == "easy":
            own_count, other_count = 6, 1
        elif difficulty == "medium":
            own_count, other_count = 4, 3
        else:
            own_count, other_count = 3, 4
        tokens = rng.sample(own_words, k=min(len(own_words), own_count))
        tokens += rng.sample(other_words, k=min(len(other_words), other_count))
        tokens += rng.sample(common_words, k=2)
        rng.shuffle(tokens)
        return " ".join(tokens)

    rows: List[Dict[str, Any]] = []
    for split_name, split_size in (("train", config["train_size"]), ("calibration", config["calibration_size"]), ("test", config["test_size"])):
        for index in range(split_size):
            label_name = rng.choice(label_names)
            difficulty = rng.choices(difficulty_names, weights=difficulty_probs, k=1)[0]
            observed_label = label_name
            if rng.random() < float(config["label_noise"]):
                observed_label = rng.choice([name for name in label_names if name != label_name])
            rows.append(
                {
                    "sample_id": f"synthetic-{split_name}-{index}",
                    "text": make_text(label_name, difficulty),
                    "label": label_to_id[observed_label],
                    "label_text": observed_label,
                    "split": split_name,
                    "source_dataset": "synthetic",
                    "difficulty": difficulty,
                }
            )
    frame = pd.DataFrame(rows)
    return frame, label_names, compute_dataset_fingerprint(frame)


def _normalize_hf_dataset_row(dataset_name: str, row: Dict[str, Any]) -> str:
    """Map one raw HF row into the single text field used by the notebook.

    Args:
        dataset_name: Registry key that decides which text fields are combined.
        row: Row dict returned by `datasets`.

    Returns:
        Normalized text string.
    """
    if dataset_name == "fancyzhx/ag_news":
        return str(row["text"])
    if dataset_name == "fancyzhx/dbpedia_14":
        return _join_non_empty([row.get("title", ""), row.get("content", "")])
    if dataset_name == "community-datasets/yahoo_answers_topics":
        return _join_non_empty(
            [row.get("question_title", ""), row.get("question_content", ""), row.get("best_answer", "")]
        )
    if dataset_name == "cornell-movie-review-data/rotten_tomatoes":
        return str(row["text"])
    if dataset_name == "SetFit/sst2":
        return str(row["text"])
    if dataset_name == "SetFit/TREC-QC":
        return str(row["text"])
    raise KeyError(f"Unsupported dataset normalization: {dataset_name}")


# All real datasets are normalized into one shared dataframe shape.
def load_real_dataset(dataset_name: str, spec: Dict[str, Any]) -> tuple[pd.DataFrame, List[str], str]:
    """Load and normalize a real Hugging Face dataset.

    Args:
        dataset_name: Hugging Face dataset id from `DATASET_REGISTRY`.
        spec: Dict with `train_split`, `calibration_split`, `test_split`, optional
            `expected_fingerprint`.

    Returns:
        `(frame, label_names, fingerprint)` in the shared notebook format.

    Notes:
        Raises if `expected_fingerprint` is set and does not match.
    """
    train_ds = load_dataset(dataset_name, split=spec["train_split"], cache_dir=str(HF_CACHE_DIR))
    calibration_ds = load_dataset(dataset_name, split=spec["calibration_split"], cache_dir=str(HF_CACHE_DIR))
    test_ds = load_dataset(dataset_name, split=spec["test_split"], cache_dir=str(HF_CACHE_DIR))
    label_column = str(spec.get("label_column", "label"))
    label_text_column = spec.get("label_text_column")
    label_feature = train_ds.features[label_column]
    label_names = list(getattr(label_feature, "names", []))
    if not label_names and label_text_column:
        id_to_name: Dict[int, str] = {}
        for dataset_split in (train_ds, calibration_ds, test_ds):
            for row in dataset_split:
                id_to_name[int(row[label_column])] = str(row[label_text_column])
        label_names = [id_to_name[index] for index in sorted(id_to_name)]
    if not label_names:
        all_label_ids = sorted(
            {
                int(row[label_column])
                for dataset_split in (train_ds, calibration_ds, test_ds)
                for row in dataset_split
            }
        )
        label_names = [str(label_id) for label_id in all_label_ids]

    rows: List[Dict[str, Any]] = []
    for split_name, dataset_split in (("train", train_ds), ("calibration", calibration_ds), ("test", test_ds)):
        for index, row in enumerate(dataset_split):
            label_id = int(row[label_column])
            label_text = label_names[label_id] if isinstance(label_names[0], str) else str(label_id)
            rows.append(
                {
                    "sample_id": f"{dataset_name}-{split_name}-{index}",
                    "text": _normalize_hf_dataset_row(dataset_name, row),
                    "label": label_id,
                    "label_text": label_text,
                    "split": split_name,
                    "source_dataset": dataset_name,
                }
            )
    frame = pd.DataFrame(rows)
    fingerprint = compute_dataset_fingerprint(frame)
    expected_fingerprint = spec.get("expected_fingerprint")
    if expected_fingerprint and expected_fingerprint != fingerprint:
        raise ValueError(
            f"Fingerprint mismatch for {dataset_name}: expected={expected_fingerprint} got={fingerprint}"
        )
    return frame, [str(name) for name in label_names], fingerprint


def load_named_dataset(dataset_name: str) -> tuple[pd.DataFrame, List[str], str]:
    """Dispatch dataset loading through the registry."""
    spec = DATASET_REGISTRY[dataset_name]
    if spec["kind"] == "synthetic":
        return build_synthetic_dataset(SYNTHETIC_CONFIG, GLOBAL_SEED)
    return load_real_dataset(dataset_name, spec)


seed_everything(GLOBAL_SEED)


In [ ]:
class EncodedTextDataset(Dataset):
    """Torch dataset wrapper over tokenizer tensors and optional labels."""
    def __init__(self, encodings: Dict[str, torch.Tensor], labels: Optional[Sequence[int]] = None) -> None:
        """Store tokenizer outputs like `input_ids` and aligned label ids."""
        self.encodings = encodings
        self.labels = list(labels) if labels is not None else None

    def __len__(self) -> int:
        """Return the number of encoded samples."""
        return int(self.encodings["input_ids"].shape[0])

    def __getitem__(self, index: int) -> Dict[str, torch.Tensor]:
        """Return one encoded sample dict, plus `labels` when present."""
        item = {key: value[index] for key, value in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[index], dtype=torch.long)
        return item


# This is the main hook for quick experiments like temperature scaling.
def postprocess_logits(logits: torch.Tensor) -> torch.Tensor:
    """Optional logit hook.

    Args:
        logits: Tensor of shape `(batch_size, num_labels)` from the classifier.

    Returns:
        Tensor with the same shape, optionally transformed before softmax.
    """
    """Меняй эту функцию в копии ноутбука, если хочешь вставить temperature scaling или другой logit hook."""
    return logits


# Minimal wrapper: enough for notebook experiments and SDK integration checks.
class TransformerTextClassifier:
    """Minimal transformer wrapper for notebook experiments and SDK checks."""
    def __init__(self, model_name: str, num_labels: int, freeze_encoder: bool = False) -> None:
        """Load tokenizer and sequence classifier.

        Args:
            model_name: Hugging Face model id.
            num_labels: Number of output classes.
            freeze_encoder: If true, leaves only the task head trainable.
        """
        self.model_name = model_name
        self.num_labels = num_labels
        self.fit_seed = GLOBAL_SEED
        self.temperature_scaler = None
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_labels,
            ignore_mismatched_sizes=True,
        ).to(DEVICE)
        self.version = 0
        if freeze_encoder and hasattr(self.model, "base_model"):
            for parameter in self.model.base_model.parameters():
                parameter.requires_grad = False

    def set_seed(self, seed: int) -> None:
        """Store the seed used by deterministic training utilities in this wrapper."""
        self.fit_seed = seed

    def _encode(self, texts: Sequence[str]) -> Dict[str, torch.Tensor]:
        """Tokenize raw texts into padded PyTorch tensors."""
        return self.tokenizer(
            list(texts),
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )

    def fit(self, texts: Sequence[str], labels: Sequence[int], **_: Any) -> None:
        """Train the classifier on aligned text and label sequences.

        Returns:
            None. Updates model weights in place and increments `version`.
        """
        if not texts:
            return
        self.model.train()
        dataset = EncodedTextDataset(self._encode(texts), labels)
        # DataLoader shuffling gets an explicit generator so repeated runs stay identical.
        loader_generator = torch.Generator()
        loader_generator.manual_seed(self.fit_seed)
        loader = DataLoader(dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, generator=loader_generator)
        optimizer = torch.optim.AdamW(
            [parameter for parameter in self.model.parameters() if parameter.requires_grad],
            lr=LEARNING_RATE,
        )
        for _epoch in range(TRAIN_EPOCHS):
            for batch in loader:
                batch = {key: value.to(DEVICE) for key, value in batch.items()}
                optimizer.zero_grad()
                output = self.model(**batch)
                output.loss.backward()
                optimizer.step()
        self.version += 1

    def _predict_logits(self, texts: Sequence[str], batch_size: int = 32) -> np.ndarray:
        """Return logits as an array with shape `(n_samples, num_labels)`."""
        if not texts:
            return np.zeros((0, self.num_labels), dtype=np.float32)
        self.model.eval()
        rows: List[np.ndarray] = []
        with torch.no_grad():
            for offset in range(0, len(texts), batch_size):
                chunk = list(texts[offset: offset + batch_size])
                encoded = {key: value.to(DEVICE) for key, value in self._encode(chunk).items()}
                logits = self.model(**encoded).logits
                logits = postprocess_logits(logits)
                if self.temperature_scaler is not None:
                    # Apply post-hoc calibration only after the model has already been trained.
                    logits = self.temperature_scaler.transform(logits)
                rows.append(logits.detach().cpu().numpy())
        return np.vstack(rows)

    def predict_proba(self, texts: Sequence[str], batch_size: int = 32) -> List[List[float]]:
        """Return nested per-class probabilities for each text sample."""
        logits = self._predict_logits(texts, batch_size=batch_size)
        if logits.size == 0:
            return []
        probabilities = torch.softmax(torch.tensor(logits), dim=-1).numpy()
        return probabilities.astype(float).tolist()

    def predict(self, texts: Sequence[str], batch_size: int = 32) -> List[int]:
        """Return argmax class ids for the provided texts."""
        probabilities = self.predict_proba(texts, batch_size=batch_size)
        return [int(np.argmax(row)) for row in probabilities]

    def evaluate(self, texts: Sequence[str], labels: Sequence[int]) -> Dict[str, float]:
        """Evaluate the classifier and return the notebook metric set."""
        predictions = self.predict(texts, batch_size=TRAIN_BATCH_SIZE)
        return {
            "accuracy": float(accuracy_score(labels, predictions)),
            "macro_f1": float(f1_score(labels, predictions, average="macro")),
        }

    def get_model_id(self) -> str:
        """Return a lightweight model identifier with an internal version tag."""
        return f"{self.model_name}:v{self.version}"


def build_model(model_key: str, num_labels: int) -> TransformerTextClassifier:
    """Instantiate a classifier from `MODEL_REGISTRY`."""
    spec = MODEL_REGISTRY[model_key]
    return TransformerTextClassifier(
        model_name=spec["model_name"],
        num_labels=num_labels,
        freeze_encoder=bool(spec.get("freeze_encoder", False)),
    )


class TemperatureScaling:
    """Post-hoc temperature scaling calibrator fit on a held-out split."""

    def __init__(self, num_bins: int = 15, scaling_mode: str = "scalar") -> None:
        if num_bins < 1:
            raise ValueError("num_bins must be at least 1")
        if scaling_mode not in {"scalar", "vector"}:
            raise ValueError("scaling_mode must be either 'scalar' or 'vector'")
        self.num_bins = num_bins
        self.scaling_mode = scaling_mode
        self.temperature: Any = 1.0
        self.is_fitted = False

    def _prepare_logits(self, logits: Any) -> torch.Tensor:
        if isinstance(logits, torch.Tensor):
            return logits.detach().clone().to(dtype=torch.float32)
        return torch.tensor(np.asarray(logits), dtype=torch.float32)

    def _prepare_labels(self, labels: Sequence[int]) -> torch.Tensor:
        return torch.tensor(np.asarray(labels), dtype=torch.long)

    def _temperature_tensor(self, device: torch.device) -> torch.Tensor:
        return torch.tensor(self.temperature, dtype=torch.float32, device=device)

    def _temperature_value(self) -> Any:
        if isinstance(self.temperature, np.ndarray):
            return self.temperature.astype(np.float64).round(6).tolist()
        return round(float(self.temperature), 6)

    def _probabilities(self, logits: Any) -> np.ndarray:
        scaled_logits = self.transform(logits)
        if isinstance(scaled_logits, torch.Tensor):
            return torch.softmax(scaled_logits, dim=-1).detach().cpu().numpy()
        return torch.softmax(torch.tensor(scaled_logits, dtype=torch.float32), dim=-1).numpy()

    def _expected_calibration_error(self, probabilities: np.ndarray, labels: np.ndarray) -> float:
        confidences = probabilities.max(axis=1)
        predictions = probabilities.argmax(axis=1)
        correctness = (predictions == labels).astype(np.float64)
        bin_edges = np.linspace(0.0, 1.0, self.num_bins + 1)
        ece = 0.0
        for bin_index in range(self.num_bins):
            left_edge = bin_edges[bin_index]
            right_edge = bin_edges[bin_index + 1]
            if bin_index == self.num_bins - 1:
                mask = (confidences >= left_edge) & (confidences <= right_edge)
            else:
                mask = (confidences >= left_edge) & (confidences < right_edge)
            if not np.any(mask):
                continue
            mean_confidence = float(confidences[mask].mean())
            mean_accuracy = float(correctness[mask].mean())
            ece += abs(mean_confidence - mean_accuracy) * float(mask.mean())
        return float(ece)
    
    def _adaptive_expected_calibration_error(self, probabilities: np.ndarray, labels: np.ndarray) -> float:
        if len(probabilities) == 0:
            return 0.0
        confidences = probabilities.max(axis=1)
        predictions = probabilities.argmax(axis=1)
        correctness = (predictions == labels).astype(np.float64)
        ece = 0.0
        sorted_indices = np.argsort(confidences)
        confidences = confidences[sorted_indices]
        correctness = correctness[sorted_indices]
        num_bins = min(self.num_bins, len(confidences))
        len_group = len(confidences) // num_bins
        for bin_index in range(num_bins):
            start_idx = bin_index * len_group
            end_idx = start_idx + len_group if bin_index < num_bins - 1 else len(confidences)
            mask = slice(start_idx, end_idx)
            group_confidences = confidences[mask]
            mean_confidence = float(group_confidences.mean())
            mean_accuracy = float(correctness[mask].mean())
            ece += abs(mean_confidence - mean_accuracy) * float(len(group_confidences) / len(confidences))
        return float(ece)

    def fit(self, logits: Any, labels: Sequence[int], max_iter: int = 50) -> Any:
        """Find scalar or vector temperature from calibration logits using LBFGS."""
        logits_tensor = self._prepare_logits(logits).to(DEVICE)
        labels_tensor = self._prepare_labels(labels).to(DEVICE)
        temperature_shape = (1,) if self.scaling_mode == "scalar" else (logits_tensor.shape[1],)
        log_temperature = torch.nn.Parameter(torch.zeros(temperature_shape, device=DEVICE))
        optimizer = torch.optim.LBFGS([log_temperature], lr=0.1, max_iter=max_iter, line_search_fn="strong_wolfe")

        def closure() -> torch.Tensor:
            optimizer.zero_grad()
            temperature = torch.exp(log_temperature)
            loss = torch.nn.functional.cross_entropy(logits_tensor / temperature, labels_tensor)
            loss.backward()
            return loss

        optimizer.step(closure)
        learned_temperature = torch.exp(log_temperature).detach().cpu().numpy()
        if self.scaling_mode == "scalar":
            self.temperature = float(learned_temperature.item())
        else:
            self.temperature = learned_temperature.astype(np.float32)
        self.is_fitted = True
        return self.temperature

    def transform(self, logits: Any) -> Any:
        """Apply the learned temperature to logits before softmax."""
        if isinstance(logits, torch.Tensor):
            temperature = self._temperature_tensor(logits.device)
            return logits / temperature
        logits_array = np.asarray(logits, dtype=np.float32)
        temperature_array = np.asarray(self.temperature, dtype=np.float32)
        return logits_array / temperature_array

    def calibration_metrics(self, logits: Any, labels: Sequence[int]) -> Dict[str, float]:
        """Return probability-quality metrics for the current temperature."""
        probabilities = self._probabilities(logits)
        labels_array = np.asarray(labels, dtype=np.int64)
        one_hot = np.eye(probabilities.shape[1], dtype=np.float64)[labels_array]
        brier = float(np.mean(np.sum((probabilities - one_hot) ** 2, axis=1)))
        accuracy = float((probabilities.argmax(axis=1) == labels_array).mean())
        return {
            "temperature": self._temperature_value(),
            "scaling_mode": self.scaling_mode,
            "nll": float(log_loss(labels_array, probabilities, labels=list(range(probabilities.shape[1])))),
            "brier": brier,
            "ece": self._expected_calibration_error(probabilities, labels_array),
            "adaptive_ece": self._adaptive_expected_calibration_error(probabilities, labels_array),
            "avg_confidence": float(probabilities.max(axis=1).mean()),
            "accuracy": accuracy,
        }


In [5]:
# Acquisition strategies only rank the unlabeled pool; training stays unchanged.
def strategy_scores(probabilities: np.ndarray, strategy_name: str) -> np.ndarray:
    """Score each sample for acquisition.

    Args:
        probabilities: Array with shape `(n_samples, num_labels)`.
        strategy_name: `random`, `entropy`, `margin`, or `least_confidence`.

    Returns:
        One score per sample; higher means stronger acquisition priority.
    """
    if strategy_name == "random":
        return np.random.random(size=len(probabilities))
    if strategy_name == "entropy":
        return -np.sum(probabilities * np.log(probabilities + 1e-12), axis=1)
    if strategy_name == "least_confidence":
        return 1.0 - probabilities.max(axis=1)
    if strategy_name == "margin":
        # `-sort(-x)` is a compact way to sort probabilities in descending order.
        ordered = -np.sort(-probabilities, axis=1)
        if ordered.shape[1] == 1:
            return 1.0 - ordered[:, 0]
        return -(ordered[:, 0] - ordered[:, 1])
    raise KeyError(f"Unsupported strategy: {strategy_name}")


def select_batch(frame: pd.DataFrame, model: TransformerTextClassifier, strategy_name: str, batch_size: int) -> pd.DataFrame:
    """Pick the next annotation batch from an unlabeled pool.

    Args:
        frame: DataFrame with at least `sample_id` and `text` columns.
        model: Model used by non-random strategies.
        strategy_name: Acquisition strategy name.
        batch_size: Maximum number of rows to return.

    Returns:
        Selected dataframe slice.
    """
    if frame.empty:
        return frame.copy()
    if strategy_name == "random":
        return frame.sample(n=min(batch_size, len(frame)), random_state=GLOBAL_SEED)
    probabilities = np.asarray(model.predict_proba(frame["text"].tolist(), batch_size=TRAIN_BATCH_SIZE))
    scores = strategy_scores(probabilities, strategy_name)
    selected = frame.assign(acquisition_score=scores).sort_values("acquisition_score", ascending=False)
    return selected.head(batch_size).drop(columns=["acquisition_score"])


# Oracle backend lets SDK mode run without any external labeling service.
class OracleLabelBackend:
    """In-memory SDK backend that returns ground-truth labels immediately."""
    def __init__(self, label_map: Dict[str, int]) -> None:
        """Store `{sample_id: label_id}` answers used as oracle annotations."""
        self.label_map = dict(label_map)
        self.ready = False

    def ensure_ready(self, label_schema: LabelSchema) -> Dict[str, Any]:
        """Validate the schema and mark the backend as ready."""
        label_schema.validate()
        self.ready = True
        return {"backend": "oracle"}

    def push_round(
        self,
        round_id: str,
        samples: Sequence[DataSample],
        prelabels: Optional[Dict[str, Any]] = None,
    ) -> RoundPushResult:
        """Register pushed samples and return synthetic task ids.

        Returns:
            `RoundPushResult` with one task id per pushed sample.
        """
        # SDK expects backend task ids, so synthesize deterministic ones from the round/sample ids.
        task_ids = {sample.sample_id: f"oracle:{round_id}:{sample.sample_id}" for sample in samples}
        return RoundPushResult(task_ids=task_ids, backend_round_ref={"round_id": round_id, "prelabels": bool(prelabels)})

    def poll_round(self, round_id: str, task_ids: Dict[str, str], policy: AnnotationPolicy) -> RoundProgress:
        """Report all current tasks as completed immediately."""
        return RoundProgress(total=len(task_ids), done=len(task_ids), ready_sample_ids=list(task_ids.keys()))

    def pull_round(self, round_id: str, task_ids: Dict[str, str]) -> RoundPullResult:
        """Return oracle annotations for every requested sample id."""
        if not self.ready:
            raise RuntimeError("OracleLabelBackend is not ready")
        now = time.time()
        annotations = {
            sample_id: [
                AnnotationRecord(
                    annotator_id="oracle",
                    created_at=now,
                    value=int(self.label_map[sample_id]),
                    score=1.0,
                )
            ]
            for sample_id in task_ids.keys()
        }
        return RoundPullResult(annotations=annotations, backend_payload={"round_id": round_id})

    def close(self) -> None:
        """Reset backend state after a run ends."""
        self.ready = False


In [6]:
# Keep one flat table so runs from copied notebooks are easy to compare later.
def append_result_row(row: Dict[str, Any]) -> None:
    """Append one experiment summary row to the shared CSV table.

    Args:
        row: Flat dict with scalar run metadata and metrics.
    """
    frame = pd.DataFrame([row])
    if RESULTS_TABLE_PATH.exists():
        existing = pd.read_csv(RESULTS_TABLE_PATH)
        frame = pd.concat([existing, frame], ignore_index=True)
    frame.to_csv(RESULTS_TABLE_PATH, index=False)


def serialize_temperature(temperature: Any) -> Any:
    """Keep scalar temperature numeric and vector temperature JSON-friendly."""
    if isinstance(temperature, np.ndarray):
        return json.dumps(temperature.astype(np.float64).round(6).tolist())
    if isinstance(temperature, (list, tuple)):
        return json.dumps(np.asarray(temperature, dtype=np.float64).round(6).tolist())
    return round(float(temperature), 6)


def make_run_id(dataset_name: str, model_key: str) -> str:
    """Build a timestamped run id from mode, dataset and model key."""
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    dataset_slug = dataset_name.replace("/", "__")
    return f"{timestamp}-{PIPELINE_MODE}-{dataset_slug}-{model_key}"


def split_frame(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Split a normalized dataset into train, calibration and test dataframes.

    Args:
        frame: DataFrame with a `split` column containing `train`, 'calibration' and `test`.

    Returns:
        `(train_frame, calibration_frame, test_frame)`.
    """
    train_frame = frame.loc[frame["split"] == "train"].reset_index(drop=True)
    calibration_frame = frame.loc[frame["split"] == "calibration"].reset_index(drop=True)
    test_frame = frame.loc[frame["split"] == "test"].reset_index(drop=True)
    if train_frame.empty or calibration_frame.empty or test_frame.empty:
        raise ValueError("Dataset must contain train, calibration and test rows")
    return train_frame, calibration_frame, test_frame


# Local mode is the fastest path for trying out model or heuristic changes.
def run_local_experiment(dataset_name: str, frame: pd.DataFrame, label_names: List[str], model_key: str, fingerprint: str) -> Dict[str, Any]:
    """Run the notebook-only active learning loop.

    Args:
        dataset_name: Dataset registry key.
        frame: Normalized dataset dataframe.
        label_names: Ordered label names used to size the classifier head.
        model_key: Model registry key.
        fingerprint: Dataset fingerprint stored in the result row.

    Returns:
        Flat dict with run metadata, final metrics and timing.
    """
    experiment_seed = make_experiment_seed(dataset_name, model_key, PIPELINE_MODE, STRATEGY_NAME)
    seed_everything(experiment_seed)
    train_frame, calibration_frame, test_frame = split_frame(frame)
    model = build_model(model_key, num_labels=len(label_names))
    model.set_seed(experiment_seed)
    labeled_frame = train_frame.iloc[0:0].copy()
    unlabeled_frame = train_frame.copy()
    round_metrics: List[Dict[str, float]] = []
    started_at = time.time()

    for _round_index in range(MAX_ROUNDS):
        if unlabeled_frame.empty:
            break
        batch_frame = select_batch(unlabeled_frame, model, STRATEGY_NAME, BATCH_SIZE)
        labeled_frame = pd.concat([labeled_frame, batch_frame], ignore_index=True)
        # Move selected rows out of the unlabeled pool after they are considered annotated.
        unlabeled_frame = unlabeled_frame.loc[~unlabeled_frame["sample_id"].isin(batch_frame["sample_id"])]
        model.fit(labeled_frame["text"].tolist(), labeled_frame["label"].tolist())
        metrics = model.evaluate(test_frame["text"].tolist(), test_frame["label"].tolist())
        round_metrics.append(metrics)

    final_metrics = round_metrics[-1] if round_metrics else {"accuracy": 0.0, "macro_f1": 0.0}
    calibration_logits = model._predict_logits(calibration_frame["text"].tolist(), batch_size=TRAIN_BATCH_SIZE)
    test_logits = model._predict_logits(test_frame["text"].tolist(), batch_size=TRAIN_BATCH_SIZE)
    temperature_scaler = TemperatureScaling(scaling_mode=CALIBRATION_MODE)
    raw_test_metrics = temperature_scaler.calibration_metrics(test_logits, test_frame["label"].tolist())
    learned_temperature = temperature_scaler.fit(calibration_logits, calibration_frame["label"].tolist())
    calibrated_test_metrics = temperature_scaler.calibration_metrics(test_logits, test_frame["label"].tolist())
    model.temperature_scaler = temperature_scaler
    return {
        "run_id": make_run_id(dataset_name, model_key),
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "note": RUN_NOTE,
        "pipeline_mode": "local",
        "dataset": dataset_name,
        "model_key": model_key,
        "model_name": MODEL_REGISTRY[model_key]["model_name"],
        "strategy": STRATEGY_NAME,
        "calibration_mode": CALIBRATION_MODE,
        "train_rows": len(train_frame),
        "calibration_rows": len(calibration_frame),
        "test_rows": len(test_frame),
        "rounds": len(round_metrics),
        "labeled_rows": len(labeled_frame),
        "accuracy": round(final_metrics["accuracy"], 6),
        "macro_f1": round(final_metrics["macro_f1"], 6),
        "temperature": serialize_temperature(learned_temperature),
        "raw_nll": round(raw_test_metrics["nll"], 6),
        "raw_brier": round(raw_test_metrics["brier"], 6),
        "raw_ece": round(raw_test_metrics["ece"], 6),
        "raw_adaptive_ece": round(raw_test_metrics["adaptive_ece"], 6),
        "calibrated_nll": round(calibrated_test_metrics["nll"], 6),
        "calibrated_brier": round(calibrated_test_metrics["brier"], 6),
        "calibrated_ece": round(calibrated_test_metrics["ece"], 6),
        "calibrated_adaptive_ece": round(calibrated_test_metrics["adaptive_ece"], 6),
        "dataset_fingerprint": fingerprint,
        "duration_seconds": round(time.time() - started_at, 2),
    }


# SDK mode checks that the same idea still works after integration.
def run_sdk_experiment(dataset_name: str, frame: pd.DataFrame, label_names: List[str], model_key: str, fingerprint: str) -> Dict[str, Any]:
    """Run one experiment through the SDK using the oracle backend.

    Args and returns:
        Same contract as `run_local_experiment`, but training and selection go
        through `ActiveLearningProject`.
    """
    experiment_seed = make_experiment_seed(dataset_name, model_key, PIPELINE_MODE, STRATEGY_NAME)
    seed_everything(experiment_seed)
    train_frame, calibration_frame, test_frame = split_frame(frame)
    model = build_model(model_key, num_labels=len(label_names))
    model.set_seed(experiment_seed)
    label_map = dict(zip(train_frame["sample_id"], train_frame["label"]))
    backend = OracleLabelBackend(label_map)
    run_id = make_run_id(dataset_name, model_key)
    workdir = SDK_RUNS_DIR / run_id
    if workdir.exists():
        # Rebuild the SDK workspace from scratch so prior state cannot leak into this run.
        shutil.rmtree(workdir)
    started_at = time.time()

    project = ActiveLearningProject(project_name=run_id, workdir=workdir)
    try:
        project.configure(
            dataset=train_frame[["sample_id", "text"]].copy(),
            model=model,
            label_schema=LabelSchema(task="text_classification", labels=label_names),
            label_backend_config=LabelBackendConfig(backend="custom"),
            scheduler_config=SchedulerConfig(mode="single", strategy=STRATEGY_NAME),
            label_backend=backend,
            annotation_policy=AnnotationPolicy(mode="latest", min_votes=1),
            cache_config=CacheConfig(enable=True, persist=False),
            fingerprint_config=FingerprintConfig(mode="fast"),
            split_config=SplitConfig(
                mode="explicit",
                # Train ids are wired explicitly because the notebook owns the test evaluation outside the SDK.
                explicit_splits={"train": train_frame["sample_id"].tolist(), "val": [], "test": []},
            ),
        )
        project.run(
            batch_size=BATCH_SIZE,
            stop_criteria=StopCriteria(max_rounds=MAX_ROUNDS),
            resume=False,
            poll_interval_seconds=0,
        )
        raw_test_logits = model._predict_logits(test_frame["text"].tolist(), batch_size=TRAIN_BATCH_SIZE)
        calibration_logits = model._predict_logits(calibration_frame["text"].tolist(), batch_size=TRAIN_BATCH_SIZE)
        temperature_scaler = TemperatureScaling(scaling_mode=CALIBRATION_MODE)
        raw_test_metrics = temperature_scaler.calibration_metrics(raw_test_logits, test_frame["label"].tolist())
        learned_temperature = temperature_scaler.fit(calibration_logits, calibration_frame["label"].tolist())
        calibrated_test_metrics = temperature_scaler.calibration_metrics(raw_test_logits, test_frame["label"].tolist())
        model.temperature_scaler = temperature_scaler
        metrics = model.evaluate(test_frame["text"].tolist(), test_frame["label"].tolist())
        status = project.status()
        labeled_rows = int(status["counts"]["labeled"])
    finally:
        project.close()
        backend.close()

    return {
        "run_id": run_id,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "note": RUN_NOTE,
        "pipeline_mode": "sdk",
        "dataset": dataset_name,
        "model_key": model_key,
        "model_name": MODEL_REGISTRY[model_key]["model_name"],
        "strategy": STRATEGY_NAME,
        "calibration_mode": CALIBRATION_MODE,
        "train_rows": len(train_frame),
        "calibration_rows": len(calibration_frame),
        "test_rows": len(test_frame),
        "rounds": MAX_ROUNDS,
        "labeled_rows": labeled_rows,
        "accuracy": round(metrics["accuracy"], 6),
        "macro_f1": round(metrics["macro_f1"], 6),
        "temperature": serialize_temperature(learned_temperature),
        "raw_nll": round(raw_test_metrics["nll"], 6),
        "raw_brier": round(raw_test_metrics["brier"], 6),
        "raw_ece": round(raw_test_metrics["ece"], 6),
        "raw_adaptive_ece": round(raw_test_metrics["adaptive_ece"], 6),
        "calibrated_nll": round(calibrated_test_metrics["nll"], 6),
        "calibrated_brier": round(calibrated_test_metrics["brier"], 6),
        "calibrated_ece": round(calibrated_test_metrics["ece"], 6),
        "calibrated_adaptive_ece": round(calibrated_test_metrics["adaptive_ece"], 6),
        "dataset_fingerprint": fingerprint,
        "duration_seconds": round(time.time() - started_at, 2),
    }


def run_experiment(dataset_name: str, frame: pd.DataFrame, label_names: List[str], model_key: str, fingerprint: str) -> Dict[str, Any]:
    """Dispatch the run to local mode or SDK mode based on config."""
    if PIPELINE_MODE == "local":
        return run_local_experiment(dataset_name, frame, label_names, model_key, fingerprint)
    return run_sdk_experiment(dataset_name, frame, label_names, model_key, fingerprint)


In [7]:
# Load once, inspect the shape, then reuse the normalized payload below.
loaded_datasets: Dict[str, Dict[str, Any]] = {}
for dataset_name in SELECTED_DATASETS:
    frame, label_names, fingerprint = load_named_dataset(dataset_name)
    loaded_datasets[dataset_name] = {
        "frame": frame,
        "label_names": label_names,
        "fingerprint": fingerprint,
    }
    print(
        {
            "dataset": dataset_name,
            "rows": len(frame),
            "train_rows": int((frame["split"] == "train").sum()),
            "calibration_rows": int((frame["split"] == "calibration").sum()),
            "test_rows": int((frame["split"] == "test").sum()),
            "labels": len(label_names),
            "fingerprint": fingerprint,
        }
    )

preview_dataset_name = SELECTED_DATASETS[0]
loaded_datasets[preview_dataset_name]["frame"].head()


{'dataset': 'synthetic', 'rows': 420, 'train_rows': 240, 'calibration_rows': 60, 'test_rows': 120, 'labels': 4, 'fingerprint': 'e9b5148f5d5180b334f5cf7eabfe8bed2ef97785'}


,sample_id,text,label,label_text,split,source_dataset,difficulty
0,synthetic-train-0,report theatre physics global experiment lab s...,2,science,train,synthetic,easy
1,synthetic-train-1,data physics playoff analysis biology discover...,2,science,train,synthetic,medium
2,synthetic-train-2,physics study music playoff theatre biology pu...,2,science,train,synthetic,medium
3,synthetic-train-3,analysis museum match festival public design c...,3,culture,train,synthetic,easy
4,synthetic-train-4,research public match coach goal season report...,0,sports,train,synthetic,easy


In [8]:
# Run the selected matrix and append one compact row per experiment.
run_rows: List[Dict[str, Any]] = []
for dataset_name in SELECTED_DATASETS:
    payload = loaded_datasets[dataset_name]
    for model_key in SELECTED_MODELS:
        print(f"running dataset={dataset_name} model={model_key} mode={PIPELINE_MODE}")
        row = run_experiment(
            dataset_name=dataset_name,
            frame=payload["frame"],
            label_names=payload["label_names"],
            model_key=model_key,
            fingerprint=payload["fingerprint"],
        )
        run_rows.append(row)
        if SAVE_RESULTS:
            append_result_row(row)

results_frame = pd.DataFrame(run_rows)
results_frame


running dataset=synthetic model=tiny_transformer mode=local


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11003.47it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\Dev\active-learning-orchestrator\.venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic alg

,run_id,timestamp,note,pipeline_mode,dataset,model_key,model_name,strategy,train_rows,calibration_rows,...,macro_f1,temperature,raw_nll,raw_brier,raw_ece,calibrated_nll,calibrated_brier,calibrated_ece,dataset_fingerprint,duration_seconds
0,20260419-234017-local-synthetic-tiny_transformer,2026-04-19 23:40:17,baseline local run,local,synthetic,tiny_transformer,distilbert-base-uncased,random,240,60,...,0.888826,1.987889,0.451842,0.182869,0.07409,0.422657,0.189814,0.093392,e9b5148f5d5180b334f5cf7eabfe8bed2ef97785,4.81


In [9]:
if SAVE_RESULTS and RESULTS_TABLE_PATH.exists():
    print(f"saved_results={RESULTS_TABLE_PATH}")
    pd.read_csv(RESULTS_TABLE_PATH).tail(len(run_rows))
else:
    print("results were not persisted")


saved_results=C:\Dev\active-learning-orchestrator\lab\experiment_runs.csv
